# ESM3-open (1.4B) scoring for decoding-design-bias

Scores every protein in `Decoding_Bias_Dataset.csv` under two modes, both using iterative masked pseudo-log-likelihood (mask one residue at a time, read native log-prob):

| Mode | Structure track | Comparable to |
| --- | --- | --- |
| `esm3_struct_cond_score` | VQ-VAE tokens from AF backbone, kept intact | PiFold, ProteinMPNN, ESM-IF |
| `esm3_seq_only_score`    | masked (no structure info)                 | ESM2-15B pppl, CARP |

The within-ESM3 struct vs no-struct contrast is the clean addition: same architecture, same tokenizer, same scoring, structure conditioning flipped on/off.

**Runtime on A100 (bf16):** ~8-12h for 7843 proteins × 2 modes. Runs with a resume-by-Entry checkpoint so disconnects are cheap.

## 1. Setup: GPU, deps, HF auth

In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q esm biopython requests tqdm

In [ ]:
# Accept the gated repo at https://huggingface.co/EvolutionaryScale/esm3-sm-open-v1 first.
from huggingface_hub import login
login()  # paste token inline

## 2. Mount Drive, clone repo, config paths

Output CSV is written to Drive so progress survives runtime disconnects. Dataset comes from the public GitHub repo (7.5MB). PDBs are fetched on-the-fly from the AlphaFold DB into ephemeral `/content/pdbs/` (they don't need to persist).

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
import os

REPO_URL = 'https://github.com/LBDillon/decoding-design-bias.git'
REPO_DIR = '/content/decoding-design-bias'
DATASET  = '/content/needs_heavy_scoring.csv'

# Drive output location - survives runtime disconnects
DRIVE_OUT_DIR = '/content/drive/MyDrive/decoding_bias_results/AF/esm3'
OUTPUT        = f'{DRIVE_OUT_DIR}/esm3_scores.csv'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

PDB_CACHE = '/content/pdbs'
os.makedirs(PDB_CACHE, exist_ok=True)
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

assert os.path.exists(DATASET), DATASET
print('output ->', OUTPUT)

## 3. Load model (≈ 1 min; 5GB download first time)

In [ ]:
import torch
from esm.models.esm3 import ESM3

assert torch.cuda.is_available(), 'Use a GPU runtime (Runtime -> Change runtime type -> GPU).'
device = torch.device('cuda')

model = ESM3.from_pretrained('esm3-open').to(device).eval()
tokenizers = model.tokenizers
mask_id = tokenizers.sequence.mask_token_id
print('mask_id =', mask_id, ' | param count:', sum(p.numel() for p in model.parameters())/1e9, 'B')

## 4. PDB fetching + scoring functions

In [ ]:
import requests

AF_URL_TEMPLATES = [
    'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v6.pdb',
    'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v4.pdb',
    'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v3.pdb',
]

def fetch_pdb(entry, cache_dir=PDB_CACHE):
    for t in AF_URL_TEMPLATES:
        url = t.format(uid=entry)
        version = url.rsplit('model_', 1)[-1].replace('.pdb', '')
        local = os.path.join(cache_dir, f'AF-{entry}-F1-model_{version}.pdb')
        if os.path.exists(local):
            return local
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                with open(local, 'wb') as fh:
                    fh.write(r.content)
                return local
        except Exception:
            continue
    return None

In [ ]:
#@title PDB-MODE (R3.3) - score on experimental PDB chains instead of AlphaFold
#@markdown Toggle ON to score the experimental-structure subset. First upload
#@markdown **pdb_scoring_inputs.csv** and unzip **pdb_chain_structs.zip** to
#@markdown `/content/`. Run this cell AFTER the config/fetch_pdb cell and BEFORE
#@markdown the validation/scoring cells. Leave OFF to use AlphaFold (default).
PDB_MODE = False  #@param {type:"boolean"}
if PDB_MODE:
    import os, pandas as pd
    DATASET = "/content/pdb_scoring_inputs.csv"   # Entry, pdb_id, pdb_chain(=A), sequence(=chain), chain_pdb_path
    assert os.path.exists(DATASET), "Upload pdb_scoring_inputs.csv to /content/"
    _CHAINDIR = "/content/pdb_chain_structs"
    _pdb_df = pd.read_csv(DATASET)
    _pmap = {r.Entry: os.path.join(_CHAINDIR, os.path.basename(str(r.chain_pdb_path)))
             for r in _pdb_df.itertuples()}
    def fetch_pdb(entry, *args, **kwargs):   # override: local single-chain experimental PDB
        p = _pmap.get(entry)
        return p if (p and os.path.exists(p)) else None
    # write/resume from a SEPARATE file so the AlphaFold run's checkpoint isn't
    # reused (otherwise resume sees the AF rows as 'already scored' -> remaining 0)
    try:
        OUTPUT = os.path.splitext(OUTPUT)[0] + "_pdb.csv"
        print("OUTPUT ->", OUTPUT)
    except NameError:
        print("WARNING: OUTPUT not defined yet - run the config cell ABOVE this one first.")
    print(f"PDB-MODE ON - {len(_pmap)} experimental-structure inputs; DATASET -> {DATASET}")
    print("'sequence' is the resolved PDB chain; structures are single-chain (chain 'A').")
    print("Scores cover the resolved region; compare to AF2 scores per-residue (R3.3).")
else:
    print("PDB-MODE OFF - using AlphaFold structures (default).")


In [ ]:
import torch.nn.functional as F
from esm.sdk.api import ESMProtein
from esm.utils.structure.protein_chain import ProteinChain
@torch.no_grad()
def score_protein(pdb_path, max_tokens_per_batch=2048):
    chain = ProteinChain.from_pdb(pdb_path)
    if not chain.sequence:
        return None

    protein = ESMProtein.from_protein_chain(chain)
    pt = model.encode(protein)

    seq_tokens = pt.sequence.to(device)
    struct_tokens = pt.structure.to(device) if pt.structure is not None else None
    L = seq_tokens.shape[0] - 2
    native = seq_tokens[1:1 + L]

    def _run(batch_size):
        results = {}
        for mode_name, use_struct in [('struct_cond', True), ('seq_only', False)]:
            lps = torch.zeros(L, device=device)
            for start in range(0, L, batch_size):
                end = min(start + batch_size, L)
                B = end - start
                seq_batch = seq_tokens.unsqueeze(0).repeat(B, 1).clone()
                for j in range(B):
                    seq_batch[j, start + j + 1] = mask_id
                struct_batch = (struct_tokens.unsqueeze(0).repeat(B, 1)
                                if use_struct and struct_tokens is not None else None)
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                    esmout = model.forward(sequence_tokens=seq_batch,
                                           structure_tokens=struct_batch)
                logits = esmout.sequence_logits.float()
                for j in range(B):
                    pos = start + j + 1
                    lp = F.log_softmax(logits[j, pos], dim=-1)
                    lps[start + j] = lp[native[start + j]]
                del esmout, logits
            results[f'{mode_name}_mean'] = lps.mean().item()
            results[f'{mode_name}_sum']  = lps.sum().item()
        return results

    batch_size = max(1, max_tokens_per_batch // (L + 2))
    while True:
        try:
            out = _run(batch_size)
            out['length'] = L
            return out
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            print(f'  OOM, retrying L={L} batch={batch_size}')

    out['length'] = L
    return out

## 5. Sanity check on 3 proteins

In [ ]:
import csv, time

with open(DATASET) as fh:
    rows = list(csv.DictReader(fh))
print('dataset:', len(rows), 'rows')

for r in rows[:3]:
    entry = r['Entry']
    t0 = time.time()
    pdb = fetch_pdb(entry)
    t_dl = time.time() - t0
    if pdb is None:
        print(entry, 'no PDB'); continue
    t0 = time.time()
    res = score_protein(pdb)
    t_sc = time.time() - t0
    print(f'{entry} L={res["length"]} struct_cond={res["struct_cond_mean"]:.4f} seq_only={res["seq_only_mean"]:.4f}  dl={t_dl:.1f}s score={t_sc:.1f}s')

## 6. Full run with resume

Safe to re-run - it picks up where it left off based on entries already in `OUTPUT`. Flushes after every protein so a disconnect only loses the in-flight one.

In [ ]:
from tqdm.auto import tqdm

from tqdm.auto import tqdm

already = set()
if os.path.exists(OUTPUT):
    with open(OUTPUT) as fh:
        for row in csv.DictReader(fh):
            already.add(row['Entry'])
    print('resuming, already scored:', len(already))

todo = [r for r in rows if r['Entry'] not in already]
todo = [
    r for r in todo
    if int(float(r.get('sequence_length') or len(r.get('sequence', '')))) <= 1500
]

print('filtered to', len(todo), 'proteins ≤1500aa')
print('remaining:', len(todo))

open_mode = 'a' if already else 'w'
with open(OUTPUT, open_mode, newline='') as out:
    w = csv.writer(out)
    if open_mode == 'w':
        w.writerow([
            'Entry', 'species', 'domain',
            'esm3_struct_cond_score', 'esm3_seq_only_score',
            'esm3_struct_cond_sum', 'esm3_seq_only_sum',
            'scored_length', 'dataset_length',
        ])

    counts = {'ok': 0, 'missing_pdb': 0, 'error': 0}
    t_start = time.time()
    for r in tqdm(todo, desc='esm3'):
        entry = r['Entry']
        pdb = fetch_pdb(entry)
        if pdb is None:
            counts['missing_pdb'] += 1
            continue
        try:
            res = score_protein(pdb)
        except Exception as exc:
            print(f'[{entry}] {exc}')
            counts['error'] += 1
            continue
        if res is None:
            counts['error'] += 1
            continue
        w.writerow([
            entry, r.get('species', ''), r.get('domain', ''),
            f"{res['struct_cond_mean']:.6f}", f"{res['seq_only_mean']:.6f}",
            f"{res['struct_cond_sum']:.6f}", f"{res['seq_only_sum']:.6f}",
            res['length'], len(r.get('sequence', '')),
        ])
        out.flush()
        torch.cuda.empty_cache()
        counts['ok'] += 1

print('done in', round(time.time() - t_start, 1), 's', counts)

## 7. Quick look at results

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT)
print(df.shape)
df[['esm3_struct_cond_score','esm3_seq_only_score']].describe()